# 01 · Entendimento do Negócio e Definição da Target

**Case Preditivo de NPS — Tech Challenge Fase 1 (Pós Tech IA para Cientistas de Dados)**

Este notebook cobre as duas primeiras etapas do desafio, seguindo a metodologia **CRISP-DM**:

1. **Entendimento do negócio** — qual problema estamos resolvendo, por que o NPS importa e quem se beneficia;
2. **Definição da target** — qual variável representa a satisfação, quando ela é coletada e quais os riscos de usá-la mal.

A etapa é propositalmente conceitual: o código aparece apenas para ilustrar pontos da discussão.

## 1. Entendimento do negócio

### 1.1 Qual problema de negócio está sendo resolvido?

A empresa cresceu rápido no e-commerce e, com mais pedidos, entregas e contatos, a experiência do cliente ficou **heterogênea**: clientes com indicadores operacionais parecidos terminam a jornada como promotores ou como detratores.

O problema tem duas camadas:

| Camada | Pergunta | Tipo de resposta |
|---|---|---|
| **Diagnóstica** | Quais fatores operacionais realmente movem a satisfação? | Análise exploratória (EDA) |
| **Preventiva** | Como identificar o cliente em risco *antes* da pesquisa e agir a tempo? | Modelo preditivo + regras de alerta |

Hoje o NPS só é medido **depois** do fim da jornada. Quando a nota chega, o estrago já aconteceu: o cliente já teve o atraso, já ligou três vezes para o SAC, já reclamou. O objetivo é **transformar o NPS de um indicador de retrovisor em um indicador de para-brisa** — usar os dados operacionais que a empresa já coleta durante a jornada para antecipar a insatisfação e priorizar ações corretivas.

### 1.2 Por que o NPS é importante para um e-commerce?

- **É um termômetro simples e comparável.** Uma única pergunta ("de 0 a 10, quanto você recomendaria?") resume a percepção do cliente sobre toda a jornada e permite comparar períodos, regiões, canais e concorrentes.
- **No e-commerce o cliente não "vê" a loja — ele vê a entrega e o atendimento.** A experiência física se resume a: o produto chegou no prazo? Chegou certo? Quando tive problema, fui bem atendido? O NPS captura exatamente essa percepção.
- **Trocar de loja custa um clique.** A barreira de troca é baixíssima; a lealdade precisa ser conquistada a cada pedido.
- **Correlaciona-se com crescimento.** Clientes promotores compram de novo, gastam mais e indicam; detratores deixam de comprar e falam mal — em público (redes sociais, sites de reclamação, avaliações de marketplace).

### 1.3 Quais áreas se beneficiam desses insights?

| Área | Como usa os insights |
|---|---|
| **Logística** | Priorizar rotas/transportadoras com mais atraso; definir SLA interno; acionar plano B quando o pedido entra em risco de atraso. |
| **Atendimento (SAC)** | Atuar proativamente com clientes em risco; reduzir recontatos; priorizar fila por risco de detração; metas de tempo de resolução. |
| **Produto / Marketplace** | Identificar produtos/sellers que geram mais reclamações; melhorar descrição e qualidade. |
| **Pricing / Comercial** | Calibrar frete, descontos e compensações (cupom, frete grátis) — e saber quando *não* adianta dar desconto. |
| **CRM / Marketing** | Campanhas de reativação para detratores previstos; programas de indicação para promotores previstos. |
| **Estratégia / Diretoria** | Acompanhar um "NPS previsto" em tempo real e priorizar investimentos onde o retorno em satisfação é maior. |

### 1.4 Reflexão: como o NPS impacta o negócio

**Recompra.** O cliente satisfeito volta. Em um e-commerce, adquirir um cliente novo (mídia paga, marketplace, cupom de primeira compra) costuma custar bem mais do que reter um existente. Promotores tendem a ter maior frequência e maior *lifetime value*; detratores tendem a não voltar. Na própria base deste case essa relação é visível: **a taxa de recompra em 30 dias é muito maior entre clientes com nota alta** (ver célula abaixo) — evidência de que satisfação e receita futura andam juntas.

**Boca a boca.** O NPS mede justamente a *intenção de recomendar*. Promotores viram canal de aquisição gratuito (indicações, avaliações positivas, conteúdo espontâneo). Detratores têm efeito simétrico e geralmente mais forte: uma experiência ruim é contada com mais frequência e hoje fica registrada publicamente (reviews, Reclame Aqui, redes sociais), afetando a conversão de quem ainda nem comprou.

**Market share.** Em um mercado com baixa barreira de troca, a participação de mercado é a soma de três efeitos que o NPS influencia: (1) **retenção** — quem fica compra mais; (2) **aquisição orgânica** — indicações reduzem o custo de aquisição; (3) **reputação** — notas e avaliações públicas influenciam a escolha entre lojas que vendem o mesmo produto. Empresas com NPS consistentemente acima dos concorrentes tendem a crescer sem precisar "comprar" cada cliente via desconto.

### 1.5 Indicadores de mercado que poderiam complementar a análise

- **Benchmarks de NPS do setor** (varejo online / marketplaces) para saber se o nosso NPS é bom ou ruim *em relação ao mercado*, e não só em relação ao nosso passado;
- **Reputação pública**: nota e índice de solução em sites de reclamação (ex.: Reclame Aqui), avaliações em marketplaces e lojas de aplicativo;
- **SLA logístico de mercado**: prazo médio de entrega praticado pelos concorrentes por região, % de entregas no prazo das transportadoras, custo de frete médio;
- **Concorrência**: preço e frete dos concorrentes para os mesmos produtos, políticas de troca/devolução, *share of wallet*;
- **Indicadores internos complementares**: taxa de recompra, churn, CAC, LTV, taxa de devolução/troca, CES (*Customer Effort Score*), volume de reclamações por mil pedidos.

In [1]:
# Configuração: permite importar o módulo src/ a partir da pasta notebooks/
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.nps_utils import load_raw_data, classify_nps, nps_index, nps_summary_by

df = load_raw_data()
df["nps_category"] = classify_nps(df["nps_score"])

# Ilustração do ponto 1.4: recompra em 30 dias por categoria de NPS
(
    df.groupby("nps_category", observed=True)["repeat_purchase_30d"]
    .mean()
    .mul(100)
    .round(1)
    .rename("% com recompra em 30 dias")
    .to_frame()
)

,% com recompra em 30 dias
nps_category,
Detrator,0.0
Neutro,38.4
Promotor,100.0


**O que a tabela mostra:** nenhum detrator recomprou em 30 dias, enquanto todos os promotores recompraram. A ligação entre satisfação e receita futura é direta. Mas o mesmo resultado acende um alerta: uma variável tão "colada" ao NPS, e medida *depois* dele, não pode ser usada para prevê-lo (ver 2.4).

## 2. Definição da target

### 2.1 Qual variável representa a satisfação do cliente?

A variável alvo é **`nps_score`**: a nota de 0 a 10 que o cliente dá à pergunta de recomendação, coletada após a experiência de compra.

A partir dela derivamos duas visões, usadas em momentos diferentes:

| Visão | Definição | Uso |
|---|---|---|
| **Nota contínua** | `nps_score` (0 a 10) | Modelo de regressão, ranking de risco, médias por segmento |
| **Categoria NPS** | Detrator (< 7) · Neutro (7 a < 9) · Promotor (≥ 9) | Cálculo do índice NPS e comunicação com o negócio |

> **Observação sobre as notas decimais.** Na metodologia clássica a nota é inteira (0-6 detrator, 7-8 neutro, 9-10 promotor). Nesta base a nota tem casas decimais (ex.: 6,9). Optamos por limites contínuos — **abaixo de 7 é detrator** — para não "promover" artificialmente clientes como 6,9 a neutros.

### 2.2 Por que ela foi escolhida?

- É **a métrica que a empresa já acompanha** e na qual a área de Experiência do Cliente é cobrada — o modelo responde à pergunta que o negócio já faz;
- Mede **lealdade/intenção de recomendação**, que é mais ligada a crescimento (recompra, indicação) do que uma satisfação pontual;
- É uma **visão do cliente**, não da operação: um pedido pode estar "dentro do SLA" e ainda assim gerar um detrator. Por isso é a melhor referência para validar quais fatores operacionais importam;
- Outras candidatas foram descartadas como target principal: **`csat_internal_score`** é um score interno (não sabemos exatamente como é calculado nem quando) e **`repeat_purchase_30d`** é um *resultado* comportamental que só se conhece 30 dias depois — ambas são mais úteis como validação do que como alvo.

### 2.3 Em que momento da jornada essa informação é coletada?

Ao **final da jornada**, depois da entrega e de eventuais interações com o atendimento:

```
Compra → Pagamento → Separação → Transporte → Entrega (atraso? tentativas?) → Atendimento/Reclamação → [Pesquisa NPS] → Recompra (30 dias)
```

É justamente esse posicionamento que motiva o case: quando a nota chega, não dá mais para evitar a experiência ruim. Todas as variáveis operacionais que aparecem *antes* da pesquisa são candidatas a "sinais de alerta".

### 2.4 Existe risco de usar essa variável de forma inadequada?

Sim — vários:

1. **Vazamento de informação (*data leakage*).** Variáveis que só existem depois da pesquisa — `repeat_purchase_30d` (medida 30 dias depois) e `csat_internal_score` (outra medida de satisfação, provavelmente coletada junto) — são altamente correlacionadas com o NPS e "inflariam" um modelo. Ele pareceria ótimo no teste e seria inútil na prática, porque essas informações não estarão disponíveis no momento em que queremos prever. **Decisão: elas ficam fora do modelo preditivo** e são usadas apenas para validação de negócio.
2. **Viés de resposta.** Nem todo cliente responde à pesquisa; quem responde costuma estar nos extremos (muito satisfeito ou muito irritado). A base pode não representar o cliente "médio".
3. **Confundir correlação com causa.** Ver que clientes com mais contatos no SAC dão notas menores não significa que "atender menos" melhora o NPS — o contato é sintoma de um problema anterior.
4. **Transformar o NPS em meta individual.** Se o NPS virar meta de bônus de equipe, cria-se incentivo para "pedir nota", filtrar quem recebe a pesquisa ou pressionar o cliente — e a métrica deixa de medir a realidade (Lei de Goodhart).
5. **Olhar só o índice agregado.** O NPS (% promotores − % detratores) esconde a distribuição: duas regiões com o mesmo NPS podem ter perfis muito diferentes. Por isso analisamos também a nota média e a % de detratores.
6. **Usar a previsão para punir ou discriminar clientes.** O modelo deve ser usado para *cuidar* de quem está em risco, nunca para dar tratamento pior a quem "provavelmente vai reclamar".

In [2]:
# Visão geral da target: distribuição das categorias e NPS da base
category_share = df["nps_category"].value_counts(normalize=True).mul(100).round(1)
print("Distribuição das categorias (%):")
print(category_share.to_string())
print(f"\nNota média: {df['nps_score'].mean():.2f}")
print(f"NPS da base: {nps_index(df['nps_score']):.1f} pontos")

Distribuição das categorias (%):
nps_category
Detrator    84.4
Neutro      11.2
Promotor     4.4

Nota média: 4.38
NPS da base: -80.0 pontos


**Leitura rápida:** a base retrata uma operação em situação crítica — a grande maioria dos pedidos termina com detratores, e o NPS é fortemente negativo. Isso reforça a urgência do problema: o desafio não é "otimizar" um NPS bom, é **estancar a geração de detratores**. Os próximos notebooks investigam de onde eles vêm.